<a href="https://colab.research.google.com/github/kartavayv/Paper-Implementations/blob/main/AlexNet/alexnet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -q kagglehub
import os
from google.colab import userdata

os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")

import kagglehub
path = kagglehub.dataset_download("ifigotin/imagenetmini-1000")
print(path)
print(os.listdir(path))

100%|██████████| 3.92G/3.92G [00:42<00:00, 99.4MB/s]

Extracting files...


/root/.cache/kagglehub/datasets/ifigotin/imagenetmini-1000/versions/1
['imagenet-mini']


In [4]:
TRAIN = path + "/imagenet-mini/train"
VAL   = path + "/imagenet-mini/val"

In [1]:
import torch as t
print(t.cuda.is_available())  # should print True

True


In [7]:
import torch as t
import torch.nn as nn


class AlexNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels=3 , out_channels=96, kernel_size=11, stride=4),
            nn.ReLU(),
            nn.LocalResponseNorm(size=5, k=2),
            nn.MaxPool2d(kernel_size=3, stride=2),

            nn.Conv2d(96, 256, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.LocalResponseNorm(size=5, k=2),
            nn.MaxPool2d(kernel_size=3, stride=2),

            nn.Conv2d(256, 384, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(384, 384, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(384, 256, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=3, stride=2),

            nn.Flatten(), #works for batches?
            nn.Dropout(0.5),
            nn.Linear(9216,4096),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(4096,4096),
            nn.ReLU(),
            nn.Linear(4096, 100)
        )


    def forward(self, X):
        return self.net(X)





In [6]:
from torchvision import datasets, transforms
import torch as t
from torch.utils.data import Dataset, DataLoader


normalize = transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])

tf_train = transforms.Compose([
    transforms.RandomResizedCrop(227),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(), normalize,
])

tf_val = transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(227),
    transforms.ToTensor(), normalize,
])


train = datasets.ImageFolder(TRAIN, tf_train)
val   = datasets.ImageFolder(VAL, tf_val)


train_loader = DataLoader(train, shuffle=True, num_workers=2, batch_size=128, pin_memory=True)
val_loader   = DataLoader(val, shuffle=False, num_workers=2, batch_size=128)

In [9]:
import torch as t
import torch.nn as nn

L_RATE = 0.01
MOMENTUM = 0.9
DECAY = 0.0005
EPOCH = 75

device = "cuda" if t.cuda.is_available() else "cpu"
model = AlexNet().to(device)

optim = t.optim.SGD(model.parameters(),lr=L_RATE, momentum=MOMENTUM, weight_decay=DECAY)
lfunc =  nn.CrossEntropyLoss()
lr_scheduler = t.optim.lr_scheduler.StepLR(optim, step_size=15, gamma=0.1)

for epoch in range(45):
    model.train()
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optim.zero_grad()
        loss = lfunc(model(x), y)
        loss.backward()
        optim.step()
    lr_scheduler.step()

    model.eval()
    correct = total = 0
    with t.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            correct += (model(x).argmax(1) == y).sum().item()
            total += y.numel()
    print(f"epoch {epoch} val acc {correct/total:.3f}")




AcceleratorError: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
